In [1]:
import faiss
import pickle
import numpy as np

In [2]:
# ---------------------------
# 1. Đọc file .txt
# ---------------------------
def load_embeddings_from_txt(path):
    names = []
    vectors = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) < 2:
                continue
            name = parts[0]
            vec = np.array(list(map(float, parts[1:])), dtype=np.float32)
            names.append(name)
            vectors.append(vec)

    vectors = np.vstack(vectors)
    return names, vectors


# ---------------------------
# 2. Xây FAISS index cosine
# ---------------------------
def build_faiss_index(txt_path, index_path, names_path):
    names, vectors = load_embeddings_from_txt(txt_path)

    # Chuẩn hóa vector để cosine = inner product
    vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)

    # Tạo index FAISS (Inner Product)
    dim = vectors.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(vectors)

    # Lưu index và tên
    faiss.write_index(index, index_path)
    with open(names_path, "wb") as f:
        pickle.dump(names, f)

    print(f"✅ Đã lưu FAISS index vào: {index_path}")
    print(f"✅ Đã lưu danh sách tên vào: {names_path}")
    print(f"📏 Tổng số vector: {len(names)} | Kích thước: {dim}")


if __name__ == "__main__":
    build_faiss_index(
        txt_path="embedded.txt",          # <-- file .txt của bạn
        index_path="embedded.faiss",      # <-- file FAISS lưu ra
        names_path="names.pkl"              # <-- file chứa danh sách tên
    )




✅ Đã lưu FAISS index vào: embedded.faiss
✅ Đã lưu danh sách tên vào: names.pkl
📏 Tổng số vector: 6 | Kích thước: 256


In [ ]:
# ---------------------------
# Load index và names
# ---------------------------
index = faiss.read_index("embedded.faiss")
with open("names.pkl", "rb") as f:
    names = pickle.load(f)

# ---------------------------
# Tìm kiếm theo tên
# ---------------------------
def search_by_name(query_name, k=5):
    # Tìm index của tên trong danh sách
    if query_name not in names:
        print(f"❌ Không tìm thấy '{query_name}' trong cơ sở dữ liệu.")
        return

    idx = names.index(query_name)

    # Lấy vector của người đó (đã normalize)
    vector = index.reconstruct(idx).reshape(1, -1)

    # Tìm k người giống nhất
    distances, indices = index.search(vector, k)

    print(f"\n🔍 Kết quả tìm giống với: {query_name}")
    for i, (ind, dist) in enumerate(zip(indices[0], distances[0])):
        print(f"{i+1}. {names[ind]}  (cosine_similarity={dist:.4f})")

# ---------------------------
# Tìm kiếm theo embedding mới
# ---------------------------
def search_by_embedding(embedding, k=5):
    # Đảm bảo embedding là numpy vector float32
    embedding = np.array(embedding, dtype=np.float32).reshape(1, -1)
    # Chuẩn hóa để tính cosine similarity
    embedding /= np.linalg.norm(embedding, axis=1, keepdims=True)

    distances, indices = index.search(embedding, k)

    print("\n🔍 Kết quả tìm giống với embedding nhập vào:")
    for i, (ind, dist) in enumerate(zip(indices[0], distances[0])):
        print(f"{i+1}. {names[ind]}  (cosine_similarity={dist:.4f})")


if __name__ == "__main__":
    search_by_name("hoang an", k=5)  # <-- thay bằng tên bạn muốn tìm



🔍 Kết quả tìm giống với: hoang an
1. hoang an  (cosine_similarity=1.0000)
2. dung  (cosine_similarity=0.8071)
3. Dung ho dang  (cosine_similarity=0.7774)
4. Quang neon  (cosine_similarity=0.6471)
5. awm  (cosine_similarity=0.5850)
